# ML Model Training: Hyperparameter Tuning with Optuna

This notebook mirrors the **Model Training** guide at [learnmlops.ops4life.com/guides/ml-model-training/](https://learnmlops.ops4life.com/guides/ml-model-training/).

Use Optuna to search the hyperparameter space of a Gradient Boosting classifier, log every trial to MLflow, and identify the best configuration.

**What we'll cover:**
1. Load training and validation data
2. Define the Optuna objective function
3. Run the hyperparameter search
4. Compare trials in MLflow

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score
import optuna
import mlflow
import mlflow.sklearn

mlflow.set_tracking_uri('http://mlflow:5000')
mlflow.set_experiment('employee-attrition-tuning')

# Silence Optuna's verbose logs
optuna.logging.set_verbosity(optuna.logging.WARNING)

## Step 1: Load training and validation data

In [ ]:
train_df = pd.read_csv('/workspace/pipeline-data/train.csv')
test_df  = pd.read_csv('/workspace/pipeline-data/test.csv')

X_train = train_df.drop(columns=['Attrition'])
y_train = train_df['Attrition']

# Use test set as validation for tuning (in production use a separate val split)
X_val = test_df.drop(columns=['Attrition'])
y_val = test_df['Attrition']

print(f'Train: {X_train.shape}  |  Val: {X_val.shape}')
print(f'Attrition rate — Train: {y_train.mean():.1%}  Val: {y_val.mean():.1%}')

## Step 2: Define the Optuna objective function

In [ ]:
def objective(trial):
    """Optuna calls this once per trial; returns the metric to maximise (ROC-AUC)."""
    params = {
        'learning_rate':   trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth':       trial.suggest_int('max_depth', 2, 8),
        'n_estimators':    trial.suggest_int('n_estimators', 50, 300),
        'subsample':       trial.suggest_float('subsample', 0.6, 1.0),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
    }

    with mlflow.start_run(run_name=f'trial-{trial.number}', nested=True):
        mlflow.log_params(params)

        clf = GradientBoostingClassifier(**params, random_state=42)
        clf.fit(X_train, y_train)

        auc = roc_auc_score(y_val, clf.predict_proba(X_val)[:, 1])
        mlflow.log_metric('val_roc_auc', auc)

    return auc

print('Objective function defined.')
print('Search space: learning_rate, max_depth, n_estimators, subsample, min_samples_split')

Optuna uses a **Tree-structured Parzen Estimator (TPE)** by default: it builds a probabilistic model of which hyperparameter regions are most promising based on past trials, then samples from those regions. This is more efficient than grid search or random search.

## Step 3: Run the hyperparameter search

In [ ]:
with mlflow.start_run(run_name='optuna-study'):
    study = optuna.create_study(
        direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=42),
        pruner=optuna.pruners.MedianPruner(),
    )
    study.optimize(objective, n_trials=20, show_progress_bar=False)

    best_params = study.best_params
    best_auc    = study.best_value

    mlflow.log_params({f'best_{k}': v for k, v in best_params.items()})
    mlflow.log_metric('best_val_roc_auc', best_auc)

print(f'\nBest ROC-AUC: {best_auc:.4f}')
print('Best params:')
for k, v in best_params.items():
    print(f'  {k}: {v}')

## Step 4: Compare trials in MLflow

Open [mlflow.learnmlops.ops4life.com](https://mlflow.learnmlops.ops4life.com) and navigate to the **employee-attrition-tuning** experiment. Each trial appears as a nested run. Use the **Chart** view to plot `val_roc_auc` against individual parameters — this reveals which hyperparameters have the most impact.

In [ ]:
# Retrain with best params on the full training set
print('Retraining with best parameters...')
best_clf = GradientBoostingClassifier(**best_params, random_state=42)
best_clf.fit(X_train, y_train)

final_auc = roc_auc_score(y_val, best_clf.predict_proba(X_val)[:, 1])
print(f'Final val ROC-AUC: {final_auc:.4f}')

## Next Steps

- Promote the best model to the MLflow registry: `02-pipeline/ml-model-training/promote.ipynb`
- Full guide: [learnmlops.ops4life.com/guides/ml-model-training/](https://learnmlops.ops4life.com/guides/ml-model-training/)